
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture — Introduction to Batch Deployment

## Overview

This lecture covers batch deployment — the most common way to put a model into production. 

This lecture covers 4 sections that build on each other:

- **A. Batch Deployment Fundamentals**. What batch deployment is, a typical use case, and the advantages and limitations of the approach.
- **B. The Batch Deployment Workflow**. The end-to-end path from a registered model to written predictions, and how to load a model from Unity Catalog with `pyfunc` and compute predictions.
- **C. Batch Inference with Feature Engineering**. Using offline feature tables and the `FeatureEngineeringClient` to run batch inference with `score_batch`, `FeatureLookup`, and `FeatureFunction`.
- **D. Performance and Optimization**. How Delta Lake features — Liquid Clustering, Predictive Optimization, `OPTIMIZE`, and `VACUUM` — keep batch inference fast and efficient.

## Learning Objectives

By the end of this lecture, you will be able to:

1. Describe batch deployment and identify scenarios in which this method is required.
2. Identify the advantages and disadvantages of deploying a model via batch processing.
3. Load a logged model from Unity Catalog using `pyfunc` and compute predictions using the `pyfunc` APIs.
4. Perform batch inference using Feature Engineering's `score_batch`, and connect Delta Lake's optimization techniques to batch deployment scenarios.

## A. Batch Deployment Fundamentals

### A1. What Is Batch Deployment

In **batch deployment**, batch processing generates predictions on a **regular schedule** and writes the results out to persistent storage to be consumed downstream (for example, ad-hoc BI). It is the **most common deployment strategy**.

Batch deployment is ideal for cases when:

- Immediate predictions are **not** necessary.
- Predictions can be made in batch fashion.
- The number/volume of (new) records to predict is **large**.
- New records arrive relatively infrequently (e.g., minutes to hours), so predictions do not need to be generated the moment data arrives.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 860 170" role="img" style="font-family: sans-serif;">
  <title>Batch Deployment at a Glance</title>

  <!-- Schedule -->
  <rect x="20" y="55" width="180" height="70" rx="10" fill="#E3F2FD" stroke="#1976D2" stroke-width="2"/>
  <text x="110" y="85" text-anchor="middle" font-size="14" font-weight="bold" fill="#1565C0">Regular schedule</text>
  <text x="110" y="106" text-anchor="middle" font-size="12" fill="#1B3139">daily / weekly / monthly</text>

  <!-- Score batch -->
  <rect x="280" y="55" width="180" height="70" rx="10" fill="#1B3139"/>
  <text x="370" y="85" text-anchor="middle" font-size="14" font-weight="bold" fill="#FFFFFF">Score a large batch</text>
  <text x="370" y="106" text-anchor="middle" font-size="12" fill="#F9F7F4">of new records at once</text>

  <!-- Persist -->
  <rect x="540" y="55" width="180" height="70" rx="10" fill="#E8F5E9" stroke="#388E3C" stroke-width="2"/>
  <text x="630" y="85" text-anchor="middle" font-size="14" font-weight="bold" fill="#2E7D32">Write to storage</text>
  <text x="630" y="106" text-anchor="middle" font-size="12" fill="#1B3139">read downstream later</text>

  <line x1="200" y1="90" x2="280" y2="90" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowA1)"/>
  <line x1="460" y1="90" x2="540" y2="90" stroke="#1B3139" stroke-width="2" marker-end="url(#arrowA1)"/>

  <defs>
    <marker id="arrowA1" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto"><polygon points="0 0, 9 3.5, 0 7" fill="#1B3139"/></marker>
  </defs>
</svg>
</div>

### A2. A Typical Batch Use Case

**Description:** Identifying potential churners in a subscription-based service.

**Scenario:**
- A machine learning model is trained on historical data — including customer usage patterns, interactions, and subscription details.
- The model is saved (registered) for later use.
- At the end of each day/week/month, the collected data for that period is processed in a batch, and churn predictions are generated.

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Batch deployment is ideal for churn prediction because actions (like retention campaigns) are scheduled, not immediate. Predictions are generated for large volumes of records, and the data does not need to be real-time—yesterday’s results are still actionable. This approach keeps costs low and simplifies operations, making it a practical fit for business processes that run on daily, weekly, or monthly cycles.
  </div>
</div>

### A3. Advantages and Limitations

Batch deployment trades freshness for simplicity and cost. Knowing both sides tells you when to reach for it — and when to choose streaming or real-time instead.

</br>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.green{--accent:#00A972;} .coral{--accent:#FF5F46;}
.ds-card-title { font-size: 16pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 14pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.tc { text-align: center; }
.dd-head { font-size: 16pt; font-weight: 700; margin: 4px 0 12px; color: var(--accent); }
</style>

<div class="ds-row">
  <div class="ds-card green">
    <div class="dd-head">ADVANTAGES</div>
    <ul>
      <li><strong>Cheapest</strong> deployment method</li>
      <li><strong>Ease</strong> of implementation</li>
      <li>Efficient per data point</li>
      <li>Can handle a <strong>high volume of data</strong></li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="dd-head">LIMITATIONS</div>
    <ul>
      <li>High <strong>latency</strong></li>
      <li><strong>Stale data</strong></li>
      <li>Not suitable for dynamic or rapidly changing data</li>
      <li>Not suitable for streaming data or real-time applications</li>
    </ul>
  </div>
</div>
</div>

## B. The Batch Deployment Workflow

### B1. The End-to-End Batch Workflow

A typical batch deployment workflow connects four stages: source data, model training, model registration, and scheduled batch inference. The whole path runs as an **automated job with Databricks Workflows**, and the predictions fan out to whatever downstream consumers need them.

<div style="max-width: 1000px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 880 340" role="img" style="font-family: sans-serif;">
  <title>A Typical Batch Model Deployment Workflow</title>

  <!-- Left column: build the model -->
  <rect x="15" y="30" width="150" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="90" y="52" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Delta Lake /</text>
  <text x="90" y="68" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Feature Engineering</text>

  <rect x="15" y="130" width="150" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="90" y="152" text-anchor="middle" font-size="12" font-weight="bold"  fill="#1B3139">Model training</text>
  <text x="90" y="169" text-anchor="middle" font-size="10" fill="#607D8B">sklearn · XGBoost · TF</text>

  <rect x="205" y="130" width="160" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="285" y="150" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Models in</text>
  <text x="285" y="165" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Unity Catalog</text>
  <text x="285" y="180" text-anchor="middle" font-size="10" fill="#607D8B">mlflow</text>

  <!-- Dashed automated-job region -->
  <rect x="400" y="20" width="465" height="300" rx="12" fill="none" stroke="#7B1FA2" stroke-width="1.8" stroke-dasharray="7,5"/>
  <text x="632" y="308" text-anchor="middle" font-size="12" font-weight="bold" fill="#6A1B9A">Automated job with Databricks Workflows</text>

  <!-- New data -->
  <rect x="440" y="35" width="150" height="60" rx="8" fill="#E8F5E9" stroke="#2E7D32" stroke-width="1.5"/>
  <text x="515" y="55" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Delta Lake /</text>
  <text x="515" y="70" text-anchor="middle" font-size="12" font-weight="bold" fill="#1B3139">Feature Engineering</text>
  <text x="515" y="88" text-anchor="middle" font-size="11" font-weight="bold" fill="#2E7D32">New Data</text>

  <!-- Batch inference -->
  <rect x="440" y="140" width="150" height="70" rx="8" fill="#FFF8E1" stroke="#F57C00" stroke-width="2"/>
  <text x="515" y="170" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">Batch Inference</text>
  <text x="515" y="192" text-anchor="middle" font-size="12" fill="#1B3139" font-family="monospace">pyfunc.predict</text>

  <!-- Downstream consumers -->
  <rect x="650" y="35" width="200" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="665" y="57" font-size="13" font-weight="bold" fill="#1B3139">Data Lake</text>
  <text x="665" y="75" font-size="11" fill="#607D8B">Offline data processing</text>

  <rect x="650" y="100" width="200" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="665" y="122" font-size="13" font-weight="bold" fill="#1B3139">Database</text>
  <text x="665" y="140" font-size="11" fill="#607D8B">Serving stored predictions</text>

  <rect x="650" y="165" width="200" height="55" rx="8" fill="#FFFFFF" stroke="#1976D2" stroke-width="1.5"/>
  <text x="665" y="187" font-size="12" font-weight="bold" fill="#1B3139">Downstream consumer</text>
  <text x="665" y="205" font-size="11" fill="#607D8B">BI, apps, reports, …</text>

  <!-- Arrows -->
  <line x1="90" y1="85" x2="90" y2="130" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowB1)"/>
  <line x1="165" y1="157" x2="205" y2="157" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowB1)"/>
  <line x1="365" y1="157" x2="440" y2="175" stroke="#1B3139" stroke-width="1.5" marker-end="url(#arrowB1)"/>
  <line x1="515" y1="95" x2="515" y2="140" stroke="#2E7D32" stroke-width="1.5" marker-end="url(#arrowB1g)"/>
  <path d="M590 168 C620 168, 625 62, 650 62" stroke="#1B3139" stroke-width="1.5" fill="none" marker-end="url(#arrowB1)"/>
  <path d="M590 172 C620 172, 625 127, 650 127" stroke="#1B3139" stroke-width="1.5" fill="none" marker-end="url(#arrowB1)"/>
  <path d="M590 178 C620 178, 625 192, 650 192" stroke="#1B3139" stroke-width="1.5" fill="none" marker-end="url(#arrowB1)"/>

  <defs>
    <marker id="arrowB1" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#1B3139"/></marker>
    <marker id="arrowB1g" markerWidth="8" markerHeight="7" refX="7" refY="3.5" orient="auto"><polygon points="0 0, 8 3.5, 0 7" fill="#2E7D32"/></marker>
  </defs>
</svg>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Batch deployment follows a repeatable workflow: train your model once, register it in Unity Catalog, and schedule a job to load the model, score new data, and write predictions to storage. This approach keeps operations simple and scalable—predictions are always available for downstream consumers like BI tools, databases, or reports, without needing real-time code or manual intervention.
  </div>
</div>

### B2. Loading a Model with `pyfunc` and Predicting

Batch inference centers on the generic **`python_function` (pyfunc)** flavor. You load a registered model from Unity Catalog by its three-level name (`catalog.schema.model`) and an alias or version, then call `.predict()` on new data — regardless of the framework the model was trained in.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>import mlflow
mlflow.set_registry_uri("databricks-uc")
</br>
<strong># Load the registered model from Unity Catalog as a generic pyfunc model</strong>
model_uri = "models:/main.ml_models.churn_model@champion"
model = mlflow.pyfunc.load_model(model_uri)
</br>
<strong># Score new data — returns a prediction per input row</strong>
predictions = model.predict(new_data_pdf)</code>
</pre>
</div>

For large tables, wrap the same model as a **Spark UDF** so scoring runs in parallel across the cluster and writes straight back to Delta:

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<strong># Distribute scoring across the cluster with a Spark UDF</strong>
<code>predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri)
scored_df = new_data_df.withColumn("prediction", predict_udf(*new_data_df.columns))
scored_df.write.mode("overwrite").saveAsTable("main.ml_models.churn_predictions")</code>
</pre>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    The <code>pyfunc</code> flavor provides a unified <code>predict()</code> interface for all registered models, regardless of their training framework (scikit-learn, XGBoost, TensorFlow, etc.). This means your batch deployment code stays simple and consistent—no need for framework-specific logic. Just load the model and call <code>predict()</code> on new data.
  </div>
</div>

## C. Batch Inference with Feature Engineering

### C1. Offline Feature Tables

When features live in **Feature Engineering in Unity Catalog**, batch inference can pull them automatically at scoring time. An **offline feature table** is simply a Delta table with a primary key, managed through the `FeatureEngineeringClient`.

**Offline Feature Tables:**
- A **Delta table with a primary key**.
- Train and serve a model for batch inference with `FeatureEngineeringClient()`.
- Support `FeatureLookup` and `FeatureFunction` for batch model inference.
- **Not optimized for real-time inference** due to latency.

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.flow-grid { display: grid; grid-template-columns: 1fr auto 1fr auto 1fr; gap: 0; align-items: stretch; }
.flow-step { background: #F9F7F4; border: 3px solid var(--accent, #4299E0); border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0; }
.flow-num { font-size: 26pt; font-weight: 800; color: var(--accent, #4299E0); line-height: 1; margin-bottom: 8px; }
.flow-label { font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35; }
.flow-arrow { display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1; }
</style>

<div class="flow-grid">
  <div class="flow-step" style="--accent:#4299E0;">
    <div class="flow-num">1</div>
    <div class="flow-label">Offline Feature Table<br><span style="font-size:12pt;font-weight:400;">Delta table + primary key</span></div>
  </div>
  <div class="flow-arrow">&rarr;</div>
  <div class="flow-step" style="--accent:#FFAB00;">
    <div class="flow-num">2</div>
    <div class="flow-label">Register lookups &amp; functions<br><span style="font-size:12pt;font-weight:400;">then train</span></div>
  </div>
  <div class="flow-arrow">&rarr;</div>
  <div class="flow-step" style="--accent:#00A972;">
    <div class="flow-num">3</div>
    <div class="flow-label">Batch Inference<br><span style="font-size:12pt;font-weight:400;">fe.score_batch</span></div>
  </div>
</div>

<svg width="100%" viewBox="0 0 860 170" role="img" style="font-family: sans-serif;">
  <!-- Client banner -->
  <rect x="230" y="50" width="400" height="40" rx="8" fill="#1B3139"/>
  <text x="430" y="75" text-anchor="middle" font-size="14" fill="#F9F7F4" font-family="monospace">fe = FeatureEngineeringClient()</text>

  <defs>
    <marker id="arrowC1" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto"><polygon points="0 0, 9 3.5, 0 7" fill="#1B3139"/></marker>
  </defs>
</svg>
</div>

### C2. Batch Inference with `score_batch`

When a model is logged with the `FeatureEngineeringClient`, the feature metadata — which features to look up and how — is packaged with the model. At inference time you pass only the **primary keys**, and `score_batch` joins the required features from the offline feature tables automatically before scoring.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient()
</br>
<strong># Inference_df needs only the primary keys — features are looked up automatically</strong>
predictions_df = fe.score_batch(
    model_uri="models:/main.ml_models.churn_model@champion",
    df=inference_df,
)
display(predictions_df)</code>
</pre>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    <strong>FeatureLookup</strong> pulls precomputed features from an offline feature table using primary keys, while <strong>FeatureFunction</strong> computes features dynamically using a Unity Catalog-governed Python UDF. Both are defined at training time and automatically applied during <code>score_batch</code>, ensuring consistent feature logic for both training and batch inference. This keeps your batch scoring reliable and repeatable, with no manual feature joins required.
  </div>
</div>

## D. Performance and Optimization

### D1. Liquid Clustering

Because batch inference reads and writes large Delta tables, the physical layout of those tables drives job performance. **Liquid clustering** is an alternative to **table partitioning** and **`ZORDER`** that simplifies data-layout decisions while optimizing query performance.

- Liquid clustering replaces partitioning and `ZORDER` — you no longer choose partition columns up front.
- Databricks **recommends liquid clustering for all new Delta tables**.
- The Databricks client manages all layout and optimization operations for the data in your table.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code>CREATE TABLE table1 (col0 int, col1 string)
USING DELTA
CLUSTER BY (col0);</code>
</pre>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Liquid clustering is available on DBR 13.3 and above. Use <code>CLUSTER BY</code> when creating Delta tables—no need to pre-select partition columns. Clustering keys can be updated later without rewriting the table, making data layout flexible and maintenance simple.
  </div>
</div>

### D2. Predictive Optimization

Rather than scheduling maintenance yourself, **Predictive Optimization** lets Databricks decide when a table needs it. Databricks **automatically identifies** tables that would benefit from maintenance operations and **runs them for you**.

- Automatically runs maintenance operations:
  - **`OPTIMIZE`** → improves query performance by optimizing file sizes.
  - **`VACUUM`** → deletes data files no longer referenced by the table.
- Predictive optimization is **enabled by default for Unity Catalog managed tables**.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code><strong>-- Check predictive optimization status (catalog)</strong>
DESCRIBE CATALOG EXTENDED &lt;catalog_name&gt;;
</br>
<strong>-- Check predictive optimization status (schema)</strong>
DESCRIBE SCHEMA EXTENDED &lt;schema_name&gt;;</code>
</pre>
</div>

### D3. `OPTIMIZE` and `VACUUM`

When you manage maintenance manually, two commands do most of the work. **`OPTIMIZE`** improves read-query speed by coalescing many small files into larger ones — directly benefiting the large scans a batch job performs. **`VACUUM`** removes data files that are no longer referenced by the table, reclaiming storage.

<div style="background: #FFF3E0; border-left: 4px solid #F57C00; border-radius: 6px; padding: 16px; margin: 12px 0; overflow-x: auto;">
<pre style="margin: 0; font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', Menlo, monospace; font-size: 15px; color: #0a0a0a; white-space: pre-wrap;">
<code><strong>-- Manually compact small files into larger ones</strong>
OPTIMIZE table1;
</br>
<strong>-- Remove data files no longer referenced by the table</strong>
VACUUM table1;</code>
</pre>
</div>

<div style="border-left: 4px solid #4299E0; background: rgba(66,153,224,0.10); padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-family: sans-serif;">
  <strong style="display:block; color:#2272B4; margin-bottom:6px; font-size:15pt;">Info</strong>
  <div style="color:#0b2026; font-size:12pt; line-height:1.55;">
    Databricks recommends enabling <strong>predictive optimization</strong> for Delta tables. This feature automatically runs <code>OPTIMIZE</code> and <code>VACUUM</code> maintenance operations, keeping your batch tables performant without manual scheduling. Manual commands are available, but predictive optimization ensures your tables stay fast and storage-efficient with minimal effort.
  </div>
</div>

## Conclusion

In this lecture, you covered batch deployment end to end:

1. **Batch Deployment Fundamentals** — Batch processing generates predictions on a regular schedule and writes them to storage for downstream use. It is the most common strategy — cheapest, easy to implement, and able to handle high data volumes — but it comes with high latency and stale data, so it fits cases where immediate predictions are not required.
2. **The Batch Deployment Workflow** — Train a model, register it in Models in Unity Catalog, then on a schedule load it with the `pyfunc` flavor and score new data — as a pandas `predict()` call or, at scale, a Spark UDF that writes results back to Delta. The whole flow runs as an automated Databricks Workflows job.
3. **Batch Inference with Feature Engineering** — Offline feature tables (Delta tables with a primary key) let the `FeatureEngineeringClient` look up features automatically. `score_batch` takes just the primary keys and joins the required features via `FeatureLookup` and `FeatureFunction`, keeping training and inference consistent.
4. **Performance and Optimization** — Delta Lake features keep batch jobs fast: Liquid Clustering replaces partitioning/`ZORDER` for data layout, Predictive Optimization auto-runs `OPTIMIZE` and `VACUUM` on UC managed tables, and those same commands can be run manually when needed.

Together, these give you a complete picture of batch deployment on Databricks — when to use it, how to run it with `pyfunc` and Feature Engineering, and how to keep the underlying Delta tables performant.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>